# 1. Setup
**Baseline 1: Independent CNNs.** Two separate CNNs. One predicts the crop, the other predicts the disease.
A pair score is log P(crop) + log P(disease), so unseen pairs get a score too.

In [1]:
import copy
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))  # lets the notebook import common/

import timm
import torch
import torch.nn as nn
import torch.nn.functional as F

from common import config
from common.config import CHECKPOINT_DIR, DEVICE, NUM_CROPS, NUM_DISEASES, set_seed
from common.dataset import get_loaders, get_pairs
from common.evaluate import evaluate, val_accuracy

MODEL_NAME = "baseline1_independent"
DEBUG = True  # True: small data, 1 epoch. Set False for the full run.
HP = {
    "backbone": "resnet50",
    "crop_lr": 1e-4,
    "disease_lr": 3e-4,
    "weight_decay": 1e-4,
    "crop_epochs": 20,
    "disease_epochs": 20,
    "batch_size": 64,
    "optimizer": "AdamW",
    "label_smoothing": 0.1,
}

crop_epochs = 1 if DEBUG else HP["crop_epochs"]
disease_epochs = 1 if DEBUG else HP["disease_epochs"]

# Single-process loading avoids Windows worker memory errors.
config.NUM_WORKERS = 0
HP["batch_size"] = 16 if DEBUG else HP["batch_size"]
HP["debug"] = DEBUG
RUN_NAME = MODEL_NAME + ("_debug" if DEBUG else "")
set_seed(42)

In [2]:
# Loader settings are configured in Setup above.


# 2. Data

In [3]:
train_loader, val_loader, seen_loader, unseen_loader = get_loaders(
    HP["batch_size"], norm="imagenet", debug=DEBUG)
# Reduce host memory pressure before any loader is iterated.
for loader in (train_loader, val_loader, seen_loader, unseen_loader):
    loader.pin_memory = False
pairs = get_pairs().to(DEVICE)

# 3. Model

In [4]:
def make_classifier(num_classes):
    return timm.create_model(HP["backbone"], pretrained=True, num_classes=num_classes)


crop_model = make_classifier(NUM_CROPS).to(DEVICE)
disease_model = make_classifier(NUM_DISEASES).to(DEVICE)


@torch.no_grad()
def predict(images):
    crop_model.eval()
    disease_model.eval()
    images = images.to(DEVICE)
    crop_log_prob = F.log_softmax(crop_model(images).float(), dim=1)
    disease_log_prob = F.log_softmax(disease_model(images).float(), dim=1)
    # Treat crop and disease as independent, so pair score = sum of log-probs
    return crop_log_prob[:, pairs[:, 0]] + disease_log_prob[:, pairs[:, 1]]

# 4. Training

In [5]:
crop_epochs = 1 if DEBUG else HP["crop_epochs"]
disease_epochs = 1 if DEBUG else HP["disease_epochs"]

crop_optimizer = torch.optim.AdamW(
    crop_model.parameters(),
    lr=HP["crop_lr"],
    weight_decay=HP["weight_decay"],
)
disease_optimizer = torch.optim.AdamW(
    disease_model.parameters(),
    lr=HP["disease_lr"],
    weight_decay=HP["weight_decay"],
)

crop_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    crop_optimizer, T_max=crop_epochs
)
disease_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    disease_optimizer, T_max=disease_epochs
)
criterion = nn.CrossEntropyLoss(label_smoothing=HP["label_smoothing"])
best_val, best_state = -1.0, None

for epoch in range(max(crop_epochs, disease_epochs)):
    train_crop = epoch < crop_epochs
    train_disease = epoch < disease_epochs

    crop_model.train(train_crop)
    disease_model.train(train_disease)

    for images, crop_labels, disease_labels, _ in train_loader:
        images = images.to(DEVICE)
        crop_labels = crop_labels.to(DEVICE)
        disease_labels = disease_labels.to(DEVICE)

        if train_crop:
            crop_optimizer.zero_grad(set_to_none=True)
            crop_loss = criterion(crop_model(images), crop_labels)
            crop_loss.backward()
            crop_optimizer.step()

        if train_disease:
            disease_optimizer.zero_grad(set_to_none=True)
            disease_loss = criterion(disease_model(images), disease_labels)
            disease_loss.backward()
            disease_optimizer.step()

    if train_crop:
        crop_scheduler.step()
    if train_disease:
        disease_scheduler.step()

    val_top1 = val_accuracy(predict, val_loader)

    print(f"epoch {epoch + 1}: crop loss {crop_loss.item():.3f}, "
          f"disease loss {disease_loss.item():.3f}, val top-1 {val_top1:.2f}%")
    if val_top1 > best_val:  # keep the best epoch by val, never by test
        best_val = val_top1
        best_state = copy.deepcopy({"crop": crop_model.state_dict(), "disease": disease_model.state_dict()})

crop_model.load_state_dict(best_state["crop"])
disease_model.load_state_dict(best_state["disease"])

epoch 1: crop loss 3.966, disease loss 4.893, val top-1 2.00%


<All keys matched successfully>

# 5. Evaluation

In [6]:
results = evaluate(predict, seen_loader, unseen_loader, RUN_NAME, HP, robust=True)

{
  "model": "baseline1_independent_debug",
  "seen_top1": 2.0,
  "seen_top5": 9.0,
  "unseen_top1": 0.0,
  "unseen_top5": 0.0,
  "harmonic_mean": 0.0,
  "crop_acc": 8.0,
  "disease_acc": 18.0,
  "robustness": {
    "brightness": {
      "top1": 2.0,
      "drop": 0.0
    },
    "blur": {
      "top1": 2.0,
      "drop": 0.0
    },
    "noise": {
      "top1": 1.0,
      "drop": 1.0
    }
  }
}


# 6. Save

In [7]:
CHECKPOINT_DIR.mkdir(exist_ok=True)
torch.save({"model_name": RUN_NAME, "state_dict": best_state, "hp": HP},
           CHECKPOINT_DIR / f"{RUN_NAME}.pt")